In [1]:
import os
import sys
import json
import asyncio
from pathlib import Path

# 1. Add project root to sys.path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from tqa.data_fetchers.fmp import FMPClient
from tqa.utils.data_formatter import format_fundamentals_for_llm
from tqa.llm.openrouter import OpenRouterClient

# Configuration
ticker = "AAPL"

# 2. Fetch data (Global variables)
client = FMPClient()
await client._get_session()
raw_ticker_data = await client.fetch_ticker_data(ticker)
await client.close()

# 3. Format & Inspect
formatted_fundamentals = format_fundamentals_for_llm(raw_ticker_data)
llm_client = OpenRouterClient()
system_prompt = llm_client.prompts_config['system_prompts']['swing_trader']
user_template = llm_client.prompts_config['user_prompts']['master_analyst']

final_user_prompt = user_template.format(
    ticker=ticker,
    fundamentals=json.dumps(formatted_fundamentals, indent=2)
)

print(f"--- PREVIEW FOR {ticker} ---")
print(final_user_prompt)


2026-05-13 10:45:16 | INFO     | base:_cleanup_old_cache:66 - Cleaning up cache files older than 7 days (cutoff: 2026-05-06)
--- PREVIEW FOR AAPL ---
Analyze the following stock: AAPL.

FUNDAMENTAL DATA PAYLOAD:
{
  "ticker": "AAPL",
  "price_performance": {
    "1D_pct": 1.82497,
    "5D_pct": 3.50678,
    "1M_pct": 15.81019,
    "3M_pct": 17.35867,
    "6M_pct": 9.97619,
    "1Y_pct": 40.97591,
    "ytd_pct": 10.41713,
    "52_week_high": 299.42,
    "52_week_low": 193.46,
    "current_price": 299.01,
    "sma_20": 278.32,
    "sma_50": 264.68,
    "sma_100": 264.85,
    "sma_200": 258.24
  },
  "recent_performance": {
    "eps_growth_qq_pct": -29.12,
    "revenue_growth_qq_pct": -22.66,
    "eps_growth_yy_pct": 22.42,
    "revenue_growth_yy_pct": 16.6
  },
  "margins_and_returns": {
    "gross_profit_margin": "46.91%",
    "operating_profit_margin": "31.97%",
    "net_profit_margin": "26.92%"
  },
  "supply_demand": {
    "float": "14.66B",
    "outstanding": "14.69B",
    "free_flo